# Hn3ml multimodel bygeb el caption ll image el 3ndna
hnst5dm  MobileNetV2 for image feature extraction  +  LSTM for text sequence learning
el dataset : Flickr30k dataset

# 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import string

import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from tensorflow.keras.preprocessing import image

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    LSTM,
    Embedding,
    Dropout,
    Add
)

from tensorflow.keras.models import load_model

import matplotlib.pyplot as plt

# 2.  Load Dataset 


In [2]:
csv_file = r"D:\programs\archive (2)\flickr30k_images\results.csv"

df = pd.read_csv(csv_file, sep='|')
df = df.sample(40000, random_state=42)

print("Dataset Size:", len(df))
print(df.head())

Dataset Size: 40000
            image_name  comment_number  \
138302  5797756884.jpg               2   
1325     109260218.jpg               0   
130789  5087543347.jpg               4   
101814   450596617.jpg               4   
22387    226481576.jpg               2   

                                                  comment  
138302   A man is sitting on an upturned white bucket ...  
1325     Spelunkers pose inside a rock cavern while ba...  
130789             A toddler enjoying her birthday cake .  
101814              Two people are walking by the ocean .  
22387    Young , smiling , blond female police officer...  


# 3. Create Full Image Path

In [3]:
df['image_path'] = r"D:\programs\archive (2)\flickr30k_images\flickr30k_images\\" + df['image_name']

print(df[['image_name', 'image_path']].head())

            image_name                                         image_path
138302  5797756884.jpg  D:\programs\archive (2)\flickr30k_images\flick...
1325     109260218.jpg  D:\programs\archive (2)\flickr30k_images\flick...
130789  5087543347.jpg  D:\programs\archive (2)\flickr30k_images\flick...
101814   450596617.jpg  D:\programs\archive (2)\flickr30k_images\flick...
22387    226481576.jpg  D:\programs\archive (2)\flickr30k_images\flick...


# 4. Check Dataset Size

In [4]:
print("Dataset Size:", len(df))
print(df.columns)

Dataset Size: 40000
Index(['image_name', ' comment_number', ' comment', 'image_path'], dtype='str')


In [5]:
df.columns = df.columns.str.strip()

print(df.columns)

Index(['image_name', 'comment_number', 'comment', 'image_path'], dtype='str')


In [6]:
# remove missing values
df = df.dropna(subset=['comment'])

# convert everything to string
df['comment'] = df['comment'].astype(str)

captions = df['comment']

print(captions.head())

138302     A man is sitting on an upturned white bucket ...
1325       Spelunkers pose inside a rock cavern while ba...
130789               A toddler enjoying her birthday cake .
101814                Two people are walking by the ocean .
22387      Young , smiling , blond female police officer...
Name: comment, dtype: str


# 5. Text Processing

In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

captions = df['comment'].astype(str)

tokenizer = Tokenizer(num_words=5000, oov_token="<unk>")
tokenizer.fit_on_texts(captions)

sequences = tokenizer.texts_to_sequences(captions)

max_length = max(len(x) for x in sequences)

text_data = pad_sequences(sequences, maxlen=max_length, padding='post')

print(text_data.shape)
print("Max Length:", max_length)

(40000, 73)
Max Length: 73


# 6. Build Image Model (MobileNetV2)

In [8]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input

# image input
image_input = Input(shape=(224, 224, 3))

# pretrained model
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_tensor=image_input
)

# freeze layers
base_model.trainable = False

# extract features
x = GlobalAveragePooling2D()(base_model.output)

image_features = x

print(image_features)

C:\Users\IT STORE\AppData\Local\Temp\ipykernel_16724\1116167957.py:11: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


<KerasTensor shape=(None, 1280), dtype=float32, sparse=False, ragged=False, name=keras_tensor_154>


# 7. Build Text Model (LSTM)

In [9]:
from tensorflow.keras.layers import Embedding, LSTM

# text input
text_input = Input(shape=(max_length,))

# embedding layer
y = Embedding(
    input_dim=5000,
    output_dim=128
)(text_input)

# LSTM
text_features = LSTM(256)(y)

print(text_features)

<KerasTensor shape=(None, 256), dtype=float32, sparse=False, ragged=False, name=keras_tensor_157>


# 8. Combine Image and Text Features

In [16]:
from tensorflow.keras.layers import Concatenate, Dense, Dropout
from tensorflow.keras.models import Model

# combine features
combined = Concatenate()([image_features, text_features])

# dense layers
z = Dense(512, activation='relu')(combined)
z = Dropout(0.5)(z)

z = Dense(256, activation='relu')(z)

# output layer
output = Dense(5000, activation='softmax')(z)

# final model
multimodal_model = Model(
    inputs=[image_input, text_input],
    outputs=output
)

# compile
multimodal_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# summary
multimodal_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 5,495,496 (20.96 MB)

 Trainable params: 3,237,512 (12.35 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

# 9. Prepare Training Data

In [11]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# generator
def data_generator(dataframe, batch_size=16):

    while True:

        for start in range(0, len(dataframe), batch_size):

            end = start + batch_size
            batch_df = dataframe.iloc[start:end]

            image_batch = []
            text_batch = []
            target_batch = []

            for _, row in batch_df.iterrows():

                try:

                    # load image
                    img = load_img(
                        row['image_path'],
                        target_size=(224,224)
                    )

                    img = img_to_array(img)
                    img = preprocess_input(img)

                    # caption sequence
                    seq = tokenizer.texts_to_sequences(
                        [row['comment']]
                    )[0]

                    # create training samples
                    for i in range(1, len(seq)):

                        in_seq = seq[:i]
                        out_seq = seq[i]

                        # padding
                        in_seq = pad_sequences(
                            [in_seq],
                            maxlen=max_length,
                            padding='post'
                        )[0]

                        # one-hot target
                        out_seq = to_categorical(
                            out_seq,
                            num_classes=5000
                        )

                        image_batch.append(img)
                        text_batch.append(in_seq)
                        target_batch.append(out_seq)

                except:
                    continue

            yield (
                (
                    np.array(image_batch),
                    np.array(text_batch)
                ),
                np.array(target_batch)
            )

# 10. Split Dataset

In [12]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print(len(train_df))
print(len(test_df))

32000
8000


# Create Generators

In [13]:
batch_size = 4

train_generator = data_generator(
    train_df,
    batch_size=batch_size
)

test_generator = data_generator(
    test_df,
    batch_size=batch_size
)

steps_per_epoch = len(train_df) // batch_size
validation_steps = len(test_df) // batch_size

print("Train Steps:", steps_per_epoch)
print("Validation Steps:", validation_steps)

Train Steps: 8000
Validation Steps: 2000


In [14]:
x, y = next(train_generator)

print(x[0].shape)
print(x[1].shape)
print(y.shape)

(33, 224, 224, 3)
(33, 73)
(33, 5000)


# 11. Train Model

In [ ]:
history = multimodal_model.fit(
    train_generator,
    validation_data=test_generator,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    epochs=3
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.sequence import pad_sequences

# reverse word index
index_to_word = {
    index: word
    for word, index in tokenizer.word_index.items()
}

# generate caption
def generate_caption(model, image_path, max_length=78):

    # load image
    img = load_img(image_path, target_size=(224,224))
    img_array = img_to_array(img)
    img_array = preprocess_input(img_array)

    # add batch dimension
    img_array = np.expand_dims(img_array, axis=0)

    # start text
    caption = ""

    # generate words
    for i in range(max_length):

        # tokenize
        seq = tokenizer.texts_to_sequences([caption])[0]

        # padding
        seq = pad_sequences(
            [seq],
            maxlen=max_length,
            padding='post'
        )

        # predict next word
        prediction = multimodal_model.predict(
            [img_array, seq],
            verbose=0
        )

        predicted_index = np.argmax(prediction)

        # convert index to word
        predicted_word = index_to_word.get(predicted_index)

        if predicted_word is None:
            break

        # add word
        caption += " " + predicted_word

    return caption

# test image
test_image = df['image_path'].iloc[0]

# generate caption
generated_caption = generate_caption(
    multimodal_model,
    test_image
)

# show image
img = load_img(test_image, target_size=(224,224))

plt.imshow(img)
plt.axis('off')

print("Generated Caption:")
print(generated_caption)